In [1]:
import pathlib, re, numpy as np, pandas as pd, plotly.graph_objects as go

In [11]:
# ── 1.  explicit repo paths (edit just this line if you move the repo) ──
ROOT_DIR   = pathlib.Path(r"C:\Users\Owner\OneDrive\Documents\GitHub\odi-data-visualization")
DATA_DIR   = ROOT_DIR / "data"
STYLE_NAME = "gauges_colored_arrow"
PLOTS_DIR  = ROOT_DIR / "plots" / STYLE_NAME
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

TEST_MODE      = True      # flip to False for every gauge
TEST_DEPARTMENT= "Finance and Management"
TEST_FIRST_N   = 1         # number of gauges to build in test mode

In [12]:
# ── 2.  performance bands ────────────────────────────────────────────────
BANDS = [
    (1.00, 3.29, "#e53935"),   # red
    (3.30, 3.49, "#fb8c00"),   # orange
    (3.50, 3.79, "#fdd835"),   # yellow
    (3.80, 4.00, "#43a047"),   # green
]
def band_colour(v, default="#999999"):
    for lo, hi, col in BANDS:
        if lo <= v <= hi:
            return col
    return default

In [13]:
# ── 3.  helpers (wrap titles, safe filenames) ────────────────────────────
SAFE = re.compile(r"[^0-9A-Za-z _()\-.]+")
clean = lambda s, n=80: SAFE.sub("_", s)[:n].strip()

def wrap(text, width=35):
    words, line, lines = text.split(), "", []
    for w in words:
        if len(line) + len(w) + 1 > width:
            lines.append(line.rstrip()); line = ""
        line += w + " "
    lines.append(line.rstrip())
    return "<br>".join(lines)


NUM_RE = re.compile(r"[-+]?\d*\.?\d+")
to_num = lambda s: float(NUM_RE.search(str(s)).group()) if NUM_RE.search(str(s)) else np.nan

In [14]:
# ── 4.  load survey CSVs → long dataframe ────────────────────────────────
def load_long():
    csvs = list(DATA_DIR.glob("*.csv"))
    if not csvs:
        raise FileNotFoundError(f"No CSVs in {DATA_DIR}")

    def tidy(path):
        df = pd.read_csv(path)
        if df.columns[0] != "Outcome":
            df = df.rename(columns={df.columns[0]: "Outcome"})
        return df[df["Outcome"].str.lower().str.strip() != "outcome"]

    wide = pd.concat([tidy(p) for p in csvs], ignore_index=True)
    wide["Outcome"] = wide["Outcome"].str.strip()
    wide.columns    = wide.columns.str.strip()

    long = (wide
            .melt(id_vars="Outcome", var_name="Department", value_name="raw")
            .assign(Score=lambda d: d["raw"].apply(to_num))
            .dropna(subset=["Score"])
            .loc[lambda d: d["Score"].between(1, 4)])
    return long

long = load_long()

In [15]:
# ── 5.  add arrow pointer to a Figure ─────────────────────────────────────
def add_arrow(fig, score, color="black", width=4):
    """
    Draw a slim arrow with a filled black hub.
    Hub is centred at (0.5, 0.48) in paper coords.
    """
    θ = 180 - (score - 1) / 3 * 180           # degrees
    r = 0.9                                   # arrow length

    # shaft
    fig.add_shape(type="line",
        x0=0.5, y0=0.48,
        x1=0.5 + r*np.cos(np.radians(θ)),
        y1=0.48 + r*np.sin(np.radians(θ)),
        xref="paper", yref="paper",
        line=dict(color=color, width=width))

    # arrow head (two small fins)
    head = 0.05
    for dθ in (+5, -5):
        fig.add_shape(type="line",
            x0=0.5 + (r-head)*np.cos(np.radians(θ+dθ)),
            y0=0.48 + (r-head)*np.sin(np.radians(θ+dθ)),
            x1=0.5 + r*np.cos(np.radians(θ)),
            y1=0.48 + r*np.sin(np.radians(θ)),
            xref="paper", yref="paper",
            line=dict(color=color, width=width))

    # filled hub circle (smaller, solid)
    fig.add_shape(type="circle",
        xref="paper", yref="paper",
        x0=0.485, y0=0.465,     # adjust to move/size hub
        x1=0.515, y1=0.495,
        fillcolor=color, line_color=color, line_width=0)

In [16]:
# ── 6.  build full gauge figure ───────────────────────────────────────────
def build_gauge(score, outcome):
    steps = [{"range":[lo,hi], "color":col,
              "line":{"width":2,"color":"white"}} for lo,hi,col in BANDS]

    fig = go.Figure(go.Indicator(
        mode   ="gauge+number",
        value  = score,
        title  = {"text": wrap(outcome), "font":{"size":16,"color":"#1f3d5a"}},
        number = {"font":{"size":38,"color":"#1f3d5a"}},
        domain = {"x":[0,1], "y":[0,0.95]},   # pushes whole indicator down
        gauge  = {
            "axis":{"range":[1,4],
                    "tickmode":"array",
                    "tickvals":np.linspace(1,4,7),
                    "tickfont":{"size":12}},
            "steps": steps,
            "bar":  {"color":"rgba(0,0,0,0)"}  # hide default bar
        }))

    add_arrow(fig, score, color="black")
    fig.update_layout(height=450, width=450,
                      margin=dict(t=80, b=20, l=20, r=20))
    return fig

In [17]:

# ── 7.  generate and save gauges ──────────────────────────────────────────
def save_gauge(dept, outcome, score):
    dept_dir = PLOTS_DIR / clean(dept)
    dept_dir.mkdir(parents=True, exist_ok=True)

    fig = build_gauge(score, outcome)
    fname = clean(outcome)
    fig.write_html(dept_dir / f"{fname}.html", include_plotlyjs="cdn")
    fig.write_image(dept_dir / f"{fname}.png", scale=2)
    print("✓", dept, "--", fname + ".png")

if TEST_MODE:
    test_rows = (long[long["Department"].str.casefold() ==
                       TEST_DEPARTMENT.casefold()].head(TEST_FIRST_N))
    for _, row in test_rows.iterrows():
        save_gauge(TEST_DEPARTMENT, row["Outcome"], row["Score"])
else:
    for dept in long["Department"].unique():
        rows = long[long["Department"].str.casefold() == dept.casefold()]
        for _, row in rows.iterrows():
            save_gauge(dept, row["Outcome"], row["Score"])

print("\n✅ Plots saved in", PLOTS_DIR)

✓ Finance and Management -- Scale Score.png

✅ Plots saved in C:\Users\Owner\OneDrive\Documents\GitHub\odi-data-visualization\plots\gauges_colored_arrow
